### Ingest the data

### use wines-cluster

In [0]:
# try:
#     from databricks.connect import DatabricksSession
#     spark = DatabricksSession.builder.getOrCreate()
# except ImportError:
#     pass  # already available in Databricks UI

username = "doman.mat@gmail.com"
schema = "wines"
file_name = "wine_reviews-2026-04-11-23-44.csv"

file_path = f"/Volumes/workspace/default/{schema}/{file_name}"

df = spark.read.csv(file_path, header=True, inferSchema=True, multiLine=True, escape='"')

df.printSchema()

In [0]:
%skip 
import re

def clean_csv_newlines(src_path: str) -> str:
    dst_path = src_path.replace(".csv", "-cleaned.csv")
    with open(src_path, encoding="utf-8") as f:
        content = f.read()
    cleaned = re.sub(
        r'"([^"]*)"',
        lambda m: '"' + re.sub(r'[\r\n]+', ' ', m.group(1)) + '"',
        content,
        flags=re.DOTALL,
    )
    with open(dst_path, "w", encoding="utf-8", newline="") as f:
        f.write(cleaned)
    print(f"Cleaned CSV written to: {dst_path}")
    return dst_path
# no newlines in the reviews in csv now 
clean_path = clean_csv_newlines(file_path)


df = spark.read.csv(clean_path, header=True, inferSchema=True)

In [0]:
print(f"Row count: {df.count():,}")

In [0]:
display(df.limit(10))

In [0]:
display(df)

Databricks data profile. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# Save as Delta Bronze

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("wine_reviews_bronze")

print("Bronze table written.")

## Cleaning and casting 

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, expr, when, sum as spark_sum

cast_columns = [
    ("alcohol",         "double"),
    ("vintage",         "int"),
    ("case_production", "long"),
    ("retail",          "double"),
    ("rating",          "int"),
    ("bottle_size",     "double"),
]

# Build all cast transformations lazily — no Spark action yet
df_cast = df
for column, dtype in cast_columns:
    df_cast = (df_cast
        .withColumn(f"{column}_cast", expr(f"try_cast({column} as {dtype})"))
        .withColumn(f"{column}_error", when(
            col(f"{column}_cast").isNull() & col(column).isNotNull(), col(column)
        ))
        .withColumn(column, col(f"{column}_cast"))
        .drop(f"{column}_cast")
    )

# create a new feature column "is_nv" directly when casting vintage
df_cast = df_cast.withColumn("is_nv", when(col("vintage_error") == "NV", 1).otherwise(0))

# Single agg pass on df_cast — nulls_before derived as: nulls_after - failures
# (a post-cast null is either a pre-existing null or a cast failure; failures counts the latter)
agg_exprs = []
for column, _ in cast_columns:
    agg_exprs.append(spark_sum(col(column).isNull().cast("int")).alias(f"{column}_nulls_after"))
    agg_exprs.append(spark_sum(col(f"{column}_error").isNotNull().cast("int")).alias(f"{column}_failures"))

stats = df_cast.agg(*agg_exprs).collect()[0]

for column, _ in cast_columns:
    nulls_after = stats[f"{column}_nulls_after"] or 0
    failures    = stats[f"{column}_failures"]    or 0
    nulls_before = nulls_after - failures
    print(f"{column:20s}  nulls before={nulls_before:>6,}  after={nulls_after:>6,}  failures={failures:>6,}")
    if failures > 0:
        df_cast.filter(col(f"{column}_error").isNotNull()).select(f"{column}_error").distinct().show()


In [0]:
df.printSchema()